# Multimodal Skin Cancer Classification (HAM10000)

Compares an **image-only** model against an **image + metadata** multimodal model
on the real **HAM10000** dermoscopy dataset.

Runs end-to-end in Google Colab — no manual download, no Kaggle token, no login.
Use a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).

> ⚠️ Educational demo only. This is a code-structure example, **not** a
> diagnostic tool.

In [ ]:
# torch / torchvision / sklearn / matplotlib are already present in Colab.
!pip install -q -U datasets timm

import os
# HAM10000 is served through HF's Xet storage, whose CAS bridge
# (cas-bridge.xethub.hf.co) intermittently returns 403s. Forcing the classic
# LFS download path avoids that. MUST be set before importing datasets.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             roc_auc_score, balanced_accuracy_score)
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
torch.backends.cudnn.benchmark = True   # autotune conv kernels (inputs are fixed 224x224)

if torch.cuda.is_available():
    device = torch.device("cuda")                     # NVIDIA GPU (e.g. Colab T4)
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")                      # Intel Arc / iGPU (XPU build)
else:
    device = torch.device("cpu")                      # fallback — slow for ResNet50
print("Using device:", device)

## 1. Load the real HAM10000 dataset

We **stream** `marmal88/skin_cancer` from the Hugging Face Hub and materialize the
images into memory (resized, as uint8). Each record has an image plus clinical
metadata (age, sex, lesion localization) and a diagnosis `dx`.

In [ ]:
N_SAMPLES = 9577   # full HAM10000 train split; lower this for faster runs
IMG_SIZE = 224

# --- Map the 7 HAM10000 diagnoses to benign (0) vs malignant (1) -------------
# HF mirrors store `dx` either as short codes (mel, bcc, ...) OR as full words
# (melanoma, basal_cell_carcinoma, ...). We normalize and accept both forms, and
# raise on anything unrecognized so a wrong mapping can't silently label every
# lesion "benign".
def _norm(dx):
    return dx.strip().lower().replace("_", " ").replace("-", " ")

_MALIGNANT = {
    "mel", "melanoma",
    "bcc", "basal cell carcinoma",
    "akiec", "actinic keratoses",
    "actinic keratoses and intraepithelial carcinoma",
    "actinic keratoses and intraepithelial carcinomae",
}
_BENIGN = {
    "nv", "melanocytic nevi",
    "bkl", "benign keratosis like lesions", "pigmented benign keratosis",
    "df", "dermatofibroma",
    "vasc", "vascular lesions",
}

def dx_to_label(dx):
    n = _norm(dx)
    if n in _MALIGNANT:
        return 1
    if n in _BENIGN:
        return 0
    raise ValueError(f"Unmapped diagnosis {dx!r}. Add it to _MALIGNANT / _BENIGN.")

# --- Stream into memory (resized, stored as uint8 to save RAM) ----------------
stream = load_dataset("marmal88/skin_cancer", split="train", streaming=True)
stream = stream.shuffle(seed=0, buffer_size=4000)

samples = []
for row in tqdm(stream, total=N_SAMPLES, desc="loading"):
    img = row["image"].convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    samples.append({
        "image": np.asarray(img, dtype=np.uint8),   # (224, 224, 3)
        "lesion_id": row["lesion_id"],              # for leakage-free splitting
        "age": row["age"],
        "sex": row["sex"],
        "localization": row["localization"],
        "label": dx_to_label(row["dx"]),
    })
    if len(samples) >= N_SAMPLES:
        break

print(f"Done. {len(samples)} samples in memory.")

# Build categorical encoders + median age from what we actually loaded.
LOCATIONS = sorted({s["localization"] for s in samples})
SEXES = sorted({s["sex"] for s in samples})
loc_to_idx = {v: i for i, v in enumerate(LOCATIONS)}
sex_to_idx = {v: i for i, v in enumerate(SEXES)}
_ages = [s["age"] for s in samples if s["age"] is not None]
median_age = float(np.median(_ages))

labels_all = [s["label"] for s in samples]
print(f"Malignant fraction: {np.mean(labels_all):.3f}  (HAM10000 is benign-skewed)")
print("Localizations:", LOCATIONS)
print("Sex values:", SEXES)

## 2. Dataset wrapper, train/val/test split, and loaders

Three-way split: **train** fits the weights, **val** picks the best epoch and the
decision threshold, **test** is the untouched final score.

**Split is grouped by `lesion_id`** — HAM10000 has several photos of the same
lesion, so a naive per-image split leaks near-duplicates across train and test
and inflates every metric. Keeping each lesion entirely within one split is the
single most important correctness fix for an honest result.

In [ ]:
from torchvision import transforms

# Pretrained ResNet expects ImageNet-normalized inputs. Augmentation on the
# TRAIN set only fights overfitting.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

class HAMDataset(Dataset):
    """In-memory records -> (image, metadata, label). Augments when train=True."""
    def __init__(self, records, train=False):
        self.records = records
        self.tf = train_tf if train else eval_tf

    def __len__(self):
        return len(self.records)

    def _metadata(self, r):
        # age -> single normalized scalar; sex & localization -> one-hot, because
        # they are categorical (no meaningful numeric ordering).
        age = r["age"] if r["age"] is not None else median_age
        sex_oh = [0.0] * len(SEXES)
        sex_oh[sex_to_idx[r["sex"]]] = 1.0
        loc_oh = [0.0] * len(LOCATIONS)
        loc_oh[loc_to_idx[r["localization"]]] = 1.0
        return np.array([age / 100.0] + sex_oh + loc_oh, dtype=np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = np.ascontiguousarray(r["image"].astype(np.float32).transpose(2, 0, 1) / 255.0)
        img = self.tf(torch.from_numpy(img))
        return (img,
                torch.from_numpy(self._metadata(r)),
                torch.tensor(r["label"], dtype=torch.long))

# Group images by lesion, then split the LESIONS 70/15/15 — so every image of a
# given lesion stays in a single split (no train/test leakage).
from collections import defaultdict
lesion_to_idxs = defaultdict(list)
for i, s in enumerate(samples):
    lesion_to_idxs[s["lesion_id"]].append(i)

lesion_ids = list(lesion_to_idxs.keys())
rng = np.random.default_rng(0)
rng.shuffle(lesion_ids)
n_les = len(lesion_ids)
n_tr, n_va = int(0.70 * n_les), int(0.15 * n_les)

def gather(chosen_lesions):
    return [samples[i] for lid in chosen_lesions for i in lesion_to_idxs[lid]]

train_records = gather(lesion_ids[:n_tr])
val_records   = gather(lesion_ids[n_tr:n_tr + n_va])
test_records  = gather(lesion_ids[n_tr + n_va:])

train_set = HAMDataset(train_records, train=True)
val_set   = HAMDataset(val_records,   train=False)
test_set  = HAMDataset(test_records,  train=False)

# Faster input pipeline: parallel workers + pinned memory (GPU) + bigger batches.
dl_kwargs = dict(num_workers=2, pin_memory=(device.type == "cuda"),
                 persistent_workers=True)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, **dl_kwargs)
val_loader   = DataLoader(val_set,   batch_size=128, **dl_kwargs)
test_loader  = DataLoader(test_set,  batch_size=128, **dl_kwargs)

# Metadata vector length: 1 (age) + one-hot(sex) + one-hot(localization).
META_DIM = 1 + len(SEXES) + len(LOCATIONS)
print(f"{n_les} unique lesions -> "
      f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)} images")
print(f"Metadata dim: {META_DIM}")

## 3. Peek at a few real samples

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, s in zip(axes, samples[:4]):
    ax.imshow(s["image"])   # raw uint8 RGB (pre-normalization)
    ax.set_title(f"{'malignant' if s['label'] else 'benign'}\n"
                 f"age={s['age']}, {s['sex']}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Models

Both models share the same image encoder. The multimodal one adds an MLP over
the metadata and concatenates the two feature vectors before classifying.

Choose the image encoder with `IMAGE_ENCODER`:
- `"resnet50"` — pretrained ResNet50, fast and strong (default).
- `"vit"` — pretrained Vision Transformer (matches the ISEF prompt; slower, needs
  `timm`, and often **not** more accurate at this data scale — a valid thing to
  report either way).
- `"cnn"` — tiny from-scratch net that runs fine on CPU.

A GPU runtime is strongly recommended for `resnet50` / `vit`.

In [ ]:
IMAGE_ENCODER = "resnet50"   # "resnet50" (fast) | "vit" (ISEF prompt) | "cnn" (CPU-ok)
FEAT_DIM = 256                # image feature size (was 128 -- ResNet50 hands off a 2048-d
                               # vector; 128 was too tight a bottleneck before fusion)
PRETRAINED = IMAGE_ENCODER in ("resnet50", "vit")
# META_DIM (metadata vector length) is computed from the data in section 2.

class ImageEncoder(nn.Module):
    """Maps a 224x224 image to a FEAT_DIM feature vector."""
    def __init__(self, out_dim=FEAT_DIM):
        super().__init__()
        if IMAGE_ENCODER == "resnet50":
            from torchvision.models import resnet50, ResNet50_Weights
            backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            backbone.fc = nn.Linear(backbone.fc.in_features, out_dim)
            # No ReLU here (there used to be one): clamping the encoder's output to
            # >=0 right before it gets concatenated with the metadata vector throws
            # away every feature that fires negative, which is half the signal out
            # of a freshly-initialized linear layer. Let the classifier see it raw.
            self.net = backbone
        elif IMAGE_ENCODER == "vit":
            import timm
            backbone = timm.create_model("vit_base_patch16_224",
                                         pretrained=True, num_classes=out_dim)
            self.net = backbone
        else:  # "cnn" — small from-scratch net, fine on CPU
            self.net = nn.Sequential(
                nn.Conv2d(3, 16, 3, stride=2, padding=1), nn.ReLU(),   # 112
                nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),  # 56
                nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),  # 28
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(64, out_dim), nn.ReLU(),
            )

    def forward(self, x):
        return self.net(x)

class ImageOnlyModel(nn.Module):
    """Baseline: classify from the image alone."""
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder()
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(FEAT_DIM, 2))

    def forward(self, image, metadata=None):   # metadata ignored
        return self.classifier(self.encoder(image))

class MultimodalModel(nn.Module):
    """Image encoder + metadata MLP, features concatenated."""
    def __init__(self):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.meta_encoder = nn.Sequential(
            nn.Linear(META_DIM, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(FEAT_DIM + 32, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, image, metadata):
        f = self.image_encoder(image)
        m = self.meta_encoder(metadata)
        return self.classifier(torch.cat([f, m], dim=1))

## 5. Train and evaluate

In [ ]:
# HAM10000 is imbalanced, so we weight the loss by inverse class frequency —
# otherwise both models just predict "benign" for everything and recall dies.
counts = np.bincount(labels_all, minlength=2)
class_weights = torch.tensor(
    counts.sum() / (2.0 * counts), dtype=torch.float32).to(device)

EARLY_STOP = False   # OFF by default — set True to stop once val AUC plateaus

# --- Checkpointing: call save_model(...) whenever you want to keep weights -----
OUT_DIR = "outputs"                # in Colab, point this at Google Drive to persist
os.makedirs(OUT_DIR, exist_ok=True)

def save_model(model, name):
    path = os.path.join(OUT_DIR, f"{name}.pt")
    torch.save(model.state_dict(), path)
    print("saved", path)
    return path

def load_model(model, name):
    path = os.path.join(OUT_DIR, f"{name}.pt")
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    print("loaded", path)
    return model

@torch.no_grad()
def collect(model, loader):
    """Return (P(malignant) array, target array) over a loader."""
    model.eval()
    use_amp = device.type == "cuda"
    probs, tgts = [], []
    for images, meta, target in loader:
        images = images.to(device, non_blocking=True)
        meta = meta.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images, meta)
            if TTA:
                flipped = model(torch.flip(images, dims=[3]), meta)
                logits = (logits + flipped) / 2
        probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
        tgts.extend(target.numpy())
    return np.array(probs), np.array(tgts)

def metrics_at(probs, tgts, threshold=0.5):
    """Metrics for the malignant class at a given decision threshold."""
    preds = (probs >= threshold).astype(int)
    both = len(np.unique(tgts)) > 1
    return {
        "accuracy": accuracy_score(tgts, preds),
        "precision": precision_score(tgts, preds, zero_division=0),
        "recall": recall_score(tgts, preds, zero_division=0),
        "auc": roc_auc_score(tgts, probs) if both else float("nan"),
    }

TTA = True   # average each prediction with its horizontal-flip twin at eval time --
             # pure inference-time variance reduction, doesn't touch training at all.

def evaluate(model, loader, threshold=0.5):
    probs, tgts = collect(model, loader)
    return metrics_at(probs, tgts, threshold)

def param_groups(model, lr):
    """Pretrained backbone gets `lr`; the freshly-initialized head (meta encoder +
    classifier, and the swapped-in final fc) gets 5x that. One shared LR undertrains
    the random-init half and/or overcooks the pretrained half -- this is standard
    discriminative fine-tuning, not a from-scratch design."""
    encoder = getattr(model, "image_encoder", None) or getattr(model, "encoder", None)
    backbone_ids = {id(p) for p in encoder.parameters()} if (PRETRAINED and encoder) else set()
    backbone, head = [], []
    for p in model.parameters():
        (backbone if id(p) in backbone_ids else head).append(p)
    if not backbone:
        return [{"params": head, "lr": lr}]
    return [{"params": backbone, "lr": lr}, {"params": head, "lr": lr * 5}]

def train_model(model, loader, val_loader, epochs=30, patience=5):
    """Cosine LR + mixed precision; early-stop and restore the best epoch (val AUC)."""
    model.to(device)
    # Scale LR with batch size (linear scaling rule) and warm up for a few epochs.
    # The batch was bumped to 128 for speed; leaving the LR at 1e-4 quartered the
    # optimizer steps AND left the effective LR far too low, which undertrained the
    # net. Scaling LR to the batch is what recovers the lost accuracy.
    base_lr = 1e-4 if PRETRAINED else 1e-3   # gentler base when fine-tuning pretrained nets
    lr = base_lr * (loader.batch_size / 32)
    opt = torch.optim.Adam(param_groups(model, lr), weight_decay=1e-4)
    warmup = min(3, max(1, epochs // 10))
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs - warmup))],
        milestones=[warmup],
    )
    # label_smoothing softens the targets a touch — cheap regularization that pairs
    # fine with the class weights and slightly de-overconfidences the probabilities.
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    use_amp = device.type == "cuda"                 # fp16 autocast on GPU (~2x faster)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    history = {"loss": [], "accuracy": [], "precision": [], "recall": [], "auc": []}
    best_auc, best_state, waited = -1.0, None, 0

    for epoch in range(epochs):
        model.train()
        total = 0.0
        bar = tqdm(loader, desc=f"epoch {epoch+1}/{epochs}", leave=False)
        for images, meta, target in bar:
            images = images.to(device, non_blocking=True)
            meta = meta.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(images, meta), target)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += loss.item() * len(target)
            bar.set_postfix(loss=f"{loss.item():.3f}")

        avg_loss = total / len(loader.dataset)
        m = evaluate(model, val_loader)   # validation metrics @ threshold 0.5
        history["loss"].append(avg_loss)
        for k in ("accuracy", "precision", "recall", "auc"):
            history[k].append(m[k])
        print(f"  epoch {epoch+1:>2}/{epochs}  loss={avg_loss:.4f}  "
              f"val_acc={m['accuracy']:.3f}  val_rec={m['recall']:.3f}  "
              f"val_auc={m['auc']:.3f}")

        if not np.isnan(m["auc"]) and m["auc"] > best_auc:
            best_auc, best_state, waited = m["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            waited += 1
            if EARLY_STOP and waited >= patience:
                print(f"  early stop: no val-AUC gain in {patience} epochs")
                break
        sched.step()

    if best_state is not None:
        model.load_state_dict(best_state)   # roll back to the best-AUC weights
    print(f"  -> restored best epoch (val AUC = {best_auc:.3f})")
    return history

In [ ]:
print("Training image-only model:")
image_model = ImageOnlyModel()
img_history = train_model(image_model, train_loader, val_loader)

print("\nTraining multimodal model:")
multi_model = MultimodalModel()
multi_history = train_model(multi_model, train_loader, val_loader)

## 6. Training curves

Loss is the training loss; accuracy / precision / recall / AUC are on the
**validation set** after each epoch. The metric panels are fixed to a 0–100%
scale so the runs are comparable at a glance.

In [ ]:
from matplotlib.ticker import PercentFormatter

metrics = ["loss", "accuracy", "precision", "recall", "auc"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, metric in zip(axes, metrics):
    # Each model may stop at a different epoch, so give each its own x-axis.
    ax.plot(range(1, len(img_history[metric]) + 1), img_history[metric],
            "o-", label="image-only")
    ax.plot(range(1, len(multi_history[metric]) + 1), multi_history[metric],
            "s-", label="multimodal")
    ax.set_title(metric)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend()
    if metric == "loss":
        ax.set_ylim(bottom=0)                       # loss isn't a percentage
    else:
        ax.set_ylim(0, 1)                           # always 0% .. 100%
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

## 7. Final comparison (validation-tuned threshold)

Accuracy at the default 0.5 threshold understates a class-weighted model. We pick
the threshold that maximizes **balanced accuracy on validation**, then apply it
once to the held-out **test** set. This is what makes the accuracy number reflect
the model's real quality instead of an arbitrary cutoff.

In [ ]:
def tune_threshold(probs, tgts):
    """Threshold maximizing balanced accuracy on the validation set."""
    grid = np.linspace(0.05, 0.95, 91)
    scores = [balanced_accuracy_score(tgts, (probs >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(scores))])

def final_report(model):
    v_probs, v_tgts = collect(model, val_loader)
    thr = tune_threshold(v_probs, v_tgts)        # tuned on validation
    t_probs, t_tgts = collect(model, test_loader)
    m = metrics_at(t_probs, t_tgts, thr)         # applied once to test
    m["threshold"] = thr
    return m

img_final = final_report(image_model)
multi_final = final_report(multi_model)

print("Test-set metrics at the validation-tuned threshold:\n")
print(f"{'Metric':<12}{'Image-only':>12}{'Multimodal':>12}")
print("-" * 36)
for k in ["accuracy", "precision", "recall", "auc"]:
    print(f"{k:<12}{img_final[k]:>12.3f}{multi_final[k]:>12.3f}")
print(f"{'threshold':<12}{img_final['threshold']:>12.2f}{multi_final['threshold']:>12.2f}")

# --- Is the multimodal AUC gain real, or noise? Bootstrap the difference. -----
# Resample the test set 2000x with replacement and recompute the AUC gap each
# time; if the 95% interval excludes 0, the improvement is statistically solid.
img_probs, tgts = collect(image_model, test_loader)
mm_probs, _ = collect(multi_model, test_loader)

boot = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    idx = boot.integers(0, len(tgts), len(tgts))
    if len(np.unique(tgts[idx])) < 2:
        continue
    diffs.append(roc_auc_score(tgts[idx], mm_probs[idx])
                 - roc_auc_score(tgts[idx], img_probs[idx]))
diffs = np.array(diffs)
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"\nMultimodal - image-only AUC: {diffs.mean():+.3f}  "
      f"(95% CI [{lo:+.3f}, {hi:+.3f}])")
print("=> metadata helps significantly" if lo > 0
      else "=> not significant at this scale (CI includes 0)")

## 8. Save / load the trained models

Checkpointing is **manual** — run the cell below whenever you want to snapshot the
current weights. In Colab the `outputs/` folder is wiped when the runtime resets;
to keep checkpoints, mount Google Drive and set `OUT_DIR` to a Drive path:

```python
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/skin_cancer/outputs'; os.makedirs(OUT_DIR, exist_ok=True)
```

In [ ]:
# Snapshot the current weights (run any time you want to keep them).
save_model(image_model, "image_only")
save_model(multi_model, "multimodal")

# To reload later: re-create the model, then load the weights.
#   image_model = load_model(ImageOnlyModel(), "image_only")
#   multi_model = load_model(MultimodalModel(), "multimodal")

## 9. Grad-CAM — where does the model look?

Grad-CAM highlights the pixels that pushed the prediction toward *malignant*. On a
decent model the heat lands on the lesion, not the surrounding skin. (Targets the
ResNet50 backbone — keep `IMAGE_ENCODER = "resnet50"` for this.)

In [ ]:
assert IMAGE_ENCODER == "resnet50", "Grad-CAM here targets ResNet50 — set IMAGE_ENCODER='resnet50'."
JET = plt.get_cmap("jet")

class GradCAM:
    # Weight a conv feature map by the gradient of the malignant logit, ReLU, upsample.
    def __init__(self, model, target_layer):
        self.model = model
        self.acts = None
        self.grads = None
        target_layer.register_forward_hook(self._f)
        target_layer.register_full_backward_hook(self._b)
    def _f(self, m, i, o):
        self.acts = o.detach()
    def _b(self, m, gi, go):
        self.grads = go[0].detach()
    def heat(self, image, meta, cls=1):
        self.model.eval()
        img = image.unsqueeze(0).to(device)
        mt = meta.unsqueeze(0).to(device)
        out = self.model(img, mt)
        self.model.zero_grad()
        out[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * self.acts).sum(1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                                              mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def overlay(orig_uint8, cam, alpha=0.45):
    heat = (JET(cam)[..., :3] * 255).astype(np.uint8)
    return (alpha * heat + (1 - alpha) * orig_uint8).astype(np.uint8)

# Target the last conv block of the multimodal model's ResNet50 backbone.
gradcam = GradCAM(multi_model, multi_model.image_encoder.net.layer4)

# Sanity check on 4 test lesions.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    img_t, meta_t, label = test_set[col]
    raw = test_records[col]["image"]
    axes[0, col].imshow(raw)
    axes[0, col].set_title("malignant" if label == 1 else "benign", fontsize=9)
    axes[1, col].imshow(overlay(raw, gradcam.heat(img_t, meta_t)))
    axes[0, col].axis("off")
    axes[1, col].axis("off")
fig.suptitle("Top: original    Bottom: Grad-CAM (red drove 'malignant')", fontsize=11)
plt.tight_layout()
plt.show()

## 10. Export the test-case results (download)

Renders a balanced sample of held-out **test** lesions — original image, Grad-CAM
overlay, prediction vs truth, metadata, and a 2-D feature coordinate (for a scatter
plot) — into `report/` and zips it. In Colab the zip auto-downloads; drop it in a
repo later and we'll build the little write-up site around it.

In [ ]:
DISPLAY_N = 500          # how many test lesions to export (raise if you want more)
import json, shutil
from sklearn.decomposition import PCA
from PIL import Image

REPORT_DIR = "report"
IMG_DIR = os.path.join(REPORT_DIR, "img")
os.makedirs(IMG_DIR, exist_ok=True)

thr = multi_final["threshold"]                 # tuned threshold from section 7

# Balanced pick of test lesions (so it is not all benign moles).
ridx = list(range(len(test_records)))
mal = [i for i in ridx if test_records[i]["label"] == 1]
ben = [i for i in ridx if test_records[i]["label"] == 0]
pk = np.random.default_rng(0)
pk.shuffle(mal); pk.shuffle(ben)
half = DISPLAY_N // 2
chosen = mal[:half] + ben[:DISPLAY_N - half]
pk.shuffle(chosen)

rows, feats = [], []
for k, i in enumerate(tqdm(chosen, desc="rendering")):
    img_t, meta_t, label = test_set[i]
    with torch.no_grad():
        f_img = multi_model.image_encoder(img_t.unsqueeze(0).to(device))
        f_meta = multi_model.meta_encoder(meta_t.unsqueeze(0).to(device))
        fused = torch.cat([f_img, f_meta], dim=1)
        prob = float(torch.softmax(multi_model.classifier(fused).float(), 1)[0, 1])
    feats.append(fused[0].cpu().numpy())
    raw = test_records[i]["image"]
    Image.fromarray(raw).save(f"{IMG_DIR}/{k}_orig.jpg", quality=85)
    Image.fromarray(overlay(raw, gradcam.heat(img_t, meta_t))).save(f"{IMG_DIR}/{k}_cam.jpg", quality=85)
    r = test_records[i]
    pred = 1 if prob >= thr else 0
    rows.append({
        "id": k, "img": f"img/{k}_orig.jpg", "cam": f"img/{k}_cam.jpg",
        "prob": round(prob, 3),
        "pred": "malignant" if pred else "benign",
        "true": "malignant" if label == 1 else "benign",
        "correct": bool(pred == label),
        "age": r["age"], "sex": r["sex"], "loc": r["localization"],
    })

# 3-D PCA coordinates for the clickable scatter (benign vs malignant tend to
# separate). The site plots x/y in the 2-D view and x/y/z in the 3-D cloud; if z
# is ever missing it falls back to P(malignant) as the depth axis, so exporting a
# real third component here is what makes the 3-D view a genuine PCA projection.
xyz = PCA(n_components=3).fit_transform(np.array(feats))
xyz = (xyz - xyz.min(0)) / (np.ptp(xyz, axis=0) + 1e-9)
for r, (x, y, z) in zip(rows, xyz):
    r["x"], r["y"], r["z"] = round(float(x), 4), round(float(y), 4), round(float(z), 4)

with open(os.path.join(REPORT_DIR, "data.json"), "w") as f:
    json.dump(rows, f)

# Also emit data.js so index.html works from a plain file:// double-click (no
# server). Browsers block fetch() of local files, but a <script src> tag is fine.
# We ship the headline test metrics + tuned threshold alongside the rows so the
# page can print real numbers and draw the decision line on the probability bar.
meta = {
    "threshold": round(float(multi_final["threshold"]), 3),
    "test_auc": round(float(multi_final["auc"]), 3),
    "test_accuracy": round(float(multi_final["accuracy"]), 3),
    "test_recall": round(float(multi_final["recall"]), 3),
    "n_shown": len(rows),
}
with open(os.path.join(REPORT_DIR, "data.js"), "w", encoding="utf-8") as f:
    f.write("window.REPORT_META = ")
    json.dump(meta, f)
    f.write(";\nwindow.REPORT_DATA = ")
    json.dump(rows, f)
    f.write(";\n")

# --- Training history + honest model comparison for the website ---------------
# This is the image-only vs multimodal comparison that was missing before: we save
# both models' per-epoch curves AND their final test metrics, plus the natural
# (imbalanced) test distribution vs the balanced 500 shown on the site, so the
# Insights page can draw real training curves and stop hand-waving the comparison.
def _round_list(xs):
    return [round(float(v), 4) for v in xs]

training = {
    "epochs": list(range(1, len(multi_history["loss"]) + 1)),
    "image_only": {
        "loss": _round_list(img_history["loss"]),
        "val_auc": _round_list(img_history["auc"]),
        "val_acc": _round_list(img_history["accuracy"]),
        "val_recall": _round_list(img_history["recall"]),
    },
    "multimodal": {
        "loss": _round_list(multi_history["loss"]),
        "val_auc": _round_list(multi_history["auc"]),
        "val_acc": _round_list(multi_history["accuracy"]),
        "val_recall": _round_list(multi_history["recall"]),
    },
    "final": {
        "image_only": {k: round(float(img_final[k]), 4) for k in ["accuracy", "precision", "recall", "auc"]},
        "multimodal": {k: round(float(multi_final[k]), 4) for k in ["accuracy", "precision", "recall", "auc"]},
        "threshold_image": round(float(img_final["threshold"]), 3),
        "threshold_multi": round(float(multi_final["threshold"]), 3),
    },
    # Full held-out test set as it really is (HAM10000 is ~80% benign) -- metrics
    # above are computed on THIS, the natural distribution, not the balanced sample.
    "test_distribution": {
        "n": len(test_records),
        "n_malignant": int(sum(1 for r in test_records if r["label"] == 1)),
    },
    # The 500 cases exported for the scatter are force-balanced 50/50 so both
    # classes are actually visible -- accuracy on a balanced set reads LOWER than on
    # the natural 80/20 split, which is why it looks modest. AUC is unaffected.
    "display_distribution": {
        "n": len(rows),
        "n_malignant": int(sum(1 for r in rows if r["true"] == "malignant")),
    },
}
with open(os.path.join(REPORT_DIR, "training.json"), "w") as f:
    json.dump(training, f)
with open(os.path.join(REPORT_DIR, "training.js"), "w", encoding="utf-8") as f:
    f.write("window.REPORT_TRAINING = ")
    json.dump(training, f)
    f.write(";")

# --- Mined insights for the website (identical stats the Insights page reads) --
# Computed from `rows` + the tuned threshold, then written as report/insights.js
# (and .json). This used to be a separate script; folding it in here means a fresh
# export never leaves the Insights page without data again.
import statistics as _st
_THR = thr

def _acc(rs):
    return round(sum(r["correct"] for r in rs) / len(rs), 4) if rs else None
def _recall(rs):
    m = [r for r in rs if r["true"] == "malignant"]
    return round(sum(r["pred"] == "malignant" for r in m) / len(m), 4) if m else None
def _precision(rs):
    pm = [r for r in rs if r["pred"] == "malignant"]
    return round(sum(r["true"] == "malignant" for r in pm) / len(pm), 4) if pm else None
def _auc(rs):
    probs = [r["prob"] for r in rs]
    tg = [1 if r["true"] == "malignant" else 0 for r in rs]
    order = sorted(range(len(probs)), key=lambda i: probs[i])
    ranks = [0.0] * len(probs)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and probs[order[j + 1]] == probs[order[i]]:
            j += 1
        for k in range(i, j + 1):
            ranks[order[k]] = (i + j) / 2.0 + 1
        i = j + 1
    npos = sum(tg); nneg = len(tg) - npos
    if npos == 0 or nneg == 0:
        return None
    rs_pos = sum(r for r, t in zip(ranks, tg) if t == 1)
    return round((rs_pos - npos * (npos + 1) / 2.0) / (npos * nneg), 4)

from collections import defaultdict as _dd
_by_sex = _dd(list)
for r in rows: _by_sex[r["sex"]].append(r)
_sex_rows = [{"sex": k, "n": len(v), "accuracy": _acc(v), "recall": _recall(v)}
             for k, v in sorted(_by_sex.items())]

_by_loc = _dd(list)
for r in rows: _by_loc[r["loc"]].append(r)
_loc_rows = []
for k, v in sorted(_by_loc.items(), key=lambda x: -len(x[1])):
    _mn = sum(r["true"] == "malignant" for r in v)
    _loc_rows.append({"loc": k, "n": len(v), "n_malignant": _mn,
                      "malignant_rate": round(_mn / len(v), 3),
                      "accuracy": _acc(v), "recall": _recall(v),
                      "reliable": len(v) >= 15 and _mn >= 8})

_age_rows = []
for lo, hi in [(0, 30), (30, 45), (45, 60), (60, 75), (75, 200)]:
    v = [r for r in rows if r["age"] is not None and lo <= r["age"] < hi]
    _mn = sum(r["true"] == "malignant" for r in v)
    _age_rows.append({"bucket": ("75+" if hi == 200 else f"{lo}-{hi}"),
                      "n": len(v), "n_malignant": _mn,
                      "malignant_rate": round(_mn / len(v), 3) if v else None,
                      "accuracy": _acc(v), "recall": _recall(v)})

_cc = [abs(r["prob"] - _THR) for r in rows if r["correct"]]
_cw = [abs(r["prob"] - _THR) for r in rows if not r["correct"]]
_nw = sum(not r["correct"] for r in rows)
_cwn = sum(1 for r in rows if not r["correct"] and abs(r["prob"] - _THR) > 0.3)
_calibration = {"mean_confidence_correct": round(_st.mean(_cc), 3) if _cc else None,
                "mean_confidence_wrong": round(_st.mean(_cw), 3) if _cw else None,
                "n_wrong": _nw, "confidently_wrong": _cwn,
                "confidently_wrong_pct": round(100 * _cwn / _nw, 1) if _nw else None}

_fn = [r for r in rows if r["true"] == "malignant" and r["pred"] == "benign"]
_fp = [r for r in rows if r["true"] == "benign" and r["pred"] == "malignant"]
_mal_ages = [r["age"] for r in rows if r["true"] == "malignant" and r["age"] is not None]
_ben_ages = [r["age"] for r in rows if r["true"] == "benign" and r["age"] is not None]
_fn_ages = [r["age"] for r in _fn if r["age"] is not None]
_errors = {"false_negatives": len(_fn), "false_positives": len(_fp),
           "fn_mean_age": round(_st.mean(_fn_ages), 1) if _fn_ages else None,
           "malignant_mean_age": round(_st.mean(_mal_ages), 1) if _mal_ages else None,
           "benign_mean_age": round(_st.mean(_ben_ages), 1) if _ben_ages else None}

def _pick(rs, key, rev=True):
    s = sorted(rs, key=key, reverse=rev)
    return s[0] if s else None
_tp = [r for r in rows if r["true"] == "malignant" and r["pred"] == "malignant"]
_tn = [r for r in rows if r["true"] == "benign" and r["pred"] == "benign"]
_examples = {
    "confident_correct_malignant": _pick(_tp, lambda r: r["prob"]),
    "confident_correct_benign": _pick(_tn, lambda r: r["prob"], rev=False),
    "confidently_missed_malignant": _pick(_fn, lambda r: r["prob"], rev=False),
    "confidently_false_alarm": _pick(_fp, lambda r: r["prob"]),
}

insights = {
    "threshold": round(float(_THR), 3),
    "overall": {"n": len(rows), "n_malignant": sum(r["true"] == "malignant" for r in rows),
                "accuracy": _acc(rows), "recall": _recall(rows),
                "precision": _precision(rows), "auc": _auc(rows)},
    "by_sex": _sex_rows, "by_location": _loc_rows, "by_age": _age_rows,
    "calibration": _calibration, "errors": _errors, "examples": _examples,
}
with open(os.path.join(REPORT_DIR, "insights.json"), "w") as f:
    json.dump(insights, f)
with open(os.path.join(REPORT_DIR, "insights.js"), "w", encoding="utf-8") as f:
    f.write("window.REPORT_INSIGHTS = ")
    json.dump(insights, f)
    f.write(";" + chr(10))

shutil.make_archive("skin_cancer_report", "zip", REPORT_DIR)
n_correct = sum(r["correct"] for r in rows)
print(f"exported {len(rows)} cases ({n_correct} correct) -> report/ + skin_cancer_report.zip")
print("open index.html (project root) to view the scatter")
try:
    from google.colab import files
    files.download("skin_cancer_report.zip")
except Exception:
    print("Not in Colab — grab report/ or skin_cancer_report.zip from the file browser.")

### Notes & honest caveats

- **AUC is the metric to trust here.** With ~80% benign lesions, raw accuracy is
  compressed near the top and swings with the threshold. AUC (~0.90+ for a decent
  run) measures how well the model *ranks* malignant above benign, independent of
  the cutoff.
- **Threshold tuning ≠ making the model better** — it just reads it at a sensible
  operating point. The underlying separability (AUC) is unchanged.
- **Leakage-free split (done):** we split by `lesion_id`, so no lesion appears in
  both train and test. Scores are lower than a naive per-image split would give —
  that lower number is the *honest* one, and saying so is a credibility win.
- **Significance, not just point estimates:** the bootstrap CI on the AUC gap
  tells you whether the metadata genuinely helped or the difference is noise at
  this sample size. Report the CI, not just "0.91 vs 0.93".
- **Future work (v2):** ViT/CLIP image+text encoder with cross-attention,
  Grad-CAM explainability, external validation on the ISIC dataset.
- **Not a diagnostic tool.** Real use needs calibration, external validation, and
  clinical oversight.